This is the original notebook reorganised for reuse. The algorithmic structure has been preserved, but local Windows paths have been replaced with editable folder variables in Section 0. A third-party user should only need to place their own images and masks into folders named in Section 0, then run the relevant section.

In [ ]:
# Imports
# Standard scientific Python packages used throughout the notebook.
# Install requirements with: pip install -

import os
import json
import random
import glob
import math
import csv
import re

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageFont, ImageChops
from sklearn.metrics import auc

print("Working directory:", os.getcwd())

In [ ]:
# 0. USER SETTINGS.
# This notebook is written so that third-party users should only need to edit this section. The rest of the code uses these Path objects rather than hard-coded local Windows folders.

from pathlib import Path

# Folder containing this notebook, unless manually changed
PROJECT_ROOT = Path.cwd()

# Synthetic validation data
# If you want to generate the synthetic dataset using Section 1, these folders will be created automatically
SYNTHETIC_IMAGE_DIR = PROJECT_ROOT / "synthetic_data" / "images"
SYNTHETIC_MASK_DIR  = PROJECT_ROOT / "synthetic_data" / "masks"
SYNTHETIC_OUTPUT_DIR = PROJECT_ROOT / "outputs_dualmode"

# Real experimental data
# Put real clonogenic assay images here
REAL_IMAGE_DIR = PROJECT_ROOT / "data" / "real_images"

# Optional: Put manual annotation masks here if you want Dice / IoU / ROC / AUC
# Masks names should ideally match the image stem, or use a common suffix
REAL_MASK_DIR = PROJECT_ROOT / "data" / "manual_masks"

# Where all real image outputs will be saved
REAL_OUTPUT_DIR = PROJECT_ROOT / "outputs_real"

# File types accepted by the image search functions
SUPPORTED_IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

# Create output folders if they do not already exist
for folder in [SYNTHETIC_IMAGE_DIR, SYNTHETIC_MASK_DIR, SYNTHETIC_OUTPUT_DIR,
               REAL_IMAGE_DIR, REAL_MASK_DIR, REAL_OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Synthetic images:", SYNTHETIC_IMAGE_DIR)
print("Synthetic masks:", SYNTHETIC_MASK_DIR)
print("Real images:", REAL_IMAGE_DIR)
print("Manual masks:", REAL_MASK_DIR)
print("Outputs:", REAL_OUTPUT_DIR)

In [ ]:
# Helper functions for portable file handling
# These utilities replace the original filenames, they allow third-party users to place their own images and masks into the folders above without editing the analysis code
MASK_SUFFIXES = ("", "_mask", "_manual", "_binary", "_gt", "_groundtruth", "_ground_truth")

def list_image_files(folder: Path):
    # Return image files in a folder, sorted by filename
    folder = Path(folder)
    return sorted([p for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in SUPPORTED_IMAGE_EXTS])


def resolve_with_any_ext(folder: Path, stem: str):
    # Find stem and any supported image extension in folder
    folder = Path(folder)
    for ext in SUPPORTED_IMAGE_EXTS:
        candidate = folder / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def find_matching_mask(image_path: Path, mask_folder: Path):
    # Find the manual mask corresponding to an image
    image_path = Path(image_path)
    mask_folder = Path(mask_folder)
    for suffix in MASK_SUFFIXES:
        match = resolve_with_any_ext(mask_folder, image_path.stem + suffix)
        if match is not None:
            return match
    return None


def matched_real_image_mask_rows(image_folder=REAL_IMAGE_DIR, mask_folder=REAL_MASK_DIR):
    # Return [(image_stem, mask_stem), ...] for real images with masks
    rows = []
    for img_path in list_image_files(image_folder):
        mask_path = find_matching_mask(img_path, mask_folder)
        if mask_path is not None:
            rows.append((img_path.stem, mask_path.stem))
    return rows


def load_binary_mask_from_path(path: Path):
    # Read a manual or predicted mask as a binary 0 / 1 array
    path = Path(path)
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None:
        raise IOError(f"Could not read mask: {path}")
    if m.ndim == 3:
        m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m > 0).astype(np.uint8)

1. Synthetic datset generation

Generates synthetic clonogenic assay images and corresponding ground-truth masks. This section is useful for validation and stress-testing the detector under known image pertubations.

In [ ]:
# Create folders
# Create synthetic-data folders (based on Section 0 settings)
os.makedirs(SYNTHETIC_IMAGE_DIR, exist_ok=True)
os.makedirs(PROJECT_ROOT / "synthetic_data" / "colony_data", exist_ok=True)
os.makedirs(SYNTHETIC_MASK_DIR, exist_ok=True)

# Base colony image generator
def generate_colony_image(size=(512, 512), count=50, min_r=6, max_r=14):
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    colonies = []

    for _ in range(count):
        x = random.randint(max_r, size[0] - max_r)
        y = random.randint(max_r, size[1] - max_r)
        r = random.randint(min_r, max_r)
        stain = random.randint(100, 255)
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(0, 0, stain))
        colonies.append((x, y, r))

    return img, colonies

# Enhanced complex generator for base_complex.png
def generate_base_complex_image(filename, size=(512, 512), count=80, min_r=4, max_r=20):
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    colonies = []

    for _ in range(count):
        x = int(random.gauss(mu=size[0] // 2, sigma=100))
        y = int(random.gauss(mu=size[1] // 2, sigma=100))
        x = max(0, min(size[0] - 1, x))
        y = max(0, min(size[1] - 1, y))
        r = random.randint(min_r, max_r)
        stain = random.randint(80, 255)
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(0, 0, stain))
        colonies.append((x, y, r))

    img.save(f"synthetic_data/images/{filename}")
    with open(f"synthetic_data/colony_data/{filename.replace('.png', '.json')}", "w") as f:
        json.dump(colonies, f)
    save_mask(colonies, filename)

# Mask saver
def save_mask(colonies, filename, size=(512, 512)):
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    for x, y, r in colonies:
        draw.ellipse((x - r, y - r, x + r, y + r), fill=255)
    mask.save(f"synthetic_data/masks/{filename}")

# Distortion functions
def apply_blur(img):
    return img.filter(ImageFilter.GaussianBlur(radius=3))

def apply_lighting_gradient(img):
    width, height = img.size
    gradient = Image.new("L", (width, height))
    for x in range(width):
        for y in range(height):
            intensity = int(255 * ((x / width) ** 1.5))  # Non-linear gradient
            gradient.putpixel((x, y), intensity)
    gradient = gradient.convert("RGB")
    return Image.blend(img, gradient, alpha=0.5)  # Stronger lighting effect

def apply_radial_glare(img):
    width, height = img.size
    cx, cy = width // 2, height // 2
    max_dist = (cx**2 + cy**2) ** 0.5

    glare = Image.new("L", img.size, 0)
    for y in range(height):
        for x in range(width):
            dist = ((x - cx)**2 + (y - cy)**2) ** 0.5
            # Invert and scale nonlinearly
            brightness = 255 * ((1 - (dist / max_dist)) ** 2.5)  # Sharper falloff
            glare.putpixel((x, y), int(min(255, max(0, brightness))))

    glare_rgb = glare.convert("RGB")
    return Image.blend(img, glare_rgb, alpha=0.5)  # Stronger blend

def apply_grid_overlay(img):
    draw = ImageDraw.Draw(img)
    step = 50
    for x in range(0, img.width, step):
        draw.line([(x, 0), (x, img.height)], fill=(150, 150, 150), width=1)
    for y in range(0, img.height, step):
        draw.line([(0, y), (img.width, y)], fill=(150, 150, 150), width=1)
    return img

def apply_partial_occlusion(img):
    draw = ImageDraw.Draw(img)
    for _ in range(5):
        x0 = random.randint(0, img.width - 100)
        y0 = random.randint(0, img.height - 100)
        x1 = x0 + random.randint(30, 100)
        y1 = y0 + random.randint(30, 100)
        draw.rectangle([x0, y0, x1, y1], fill=(180, 180, 180))
    return img

def apply_background_noise(img):
    np_img = np.array(img)
    noise = np.random.randint(0, 30, np_img.shape, dtype='uint8')
    noisy_img = np.clip(np_img + noise, 0, 255).astype('uint8')
    return Image.fromarray(noisy_img)

def apply_artifacts(img):
    draw = ImageDraw.Draw(img)
    for _ in range(3):
        x0 = random.randint(0, img.width)
        y0 = random.randint(0, img.height)
        x1 = random.randint(0, img.width)
        y1 = random.randint(0, img.height)
        draw.line([x0, y0, x1, y1], fill=(0, 0, 0), width=2)
    draw.text((10, 10), "Label", fill=(0, 0, 0))
    return img

def apply_edge_clipping(img, colonies, size=(512, 512)):
    """Draws colonies so that some are clipped at edges."""
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    new_colonies = []

    for x, y, r in colonies:
        # Shift some colonies outside the boundary
        x += random.choice([-r * 2, 0, r * 2])
        y += random.choice([-r * 2, 0, r * 2])
        new_colonies.append((x, y, r))
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(0, 0, random.randint(100, 255)))

    return img, new_colonies

def apply_inverted_contrast(img):
    """Inverts colony contrast — bright colonies on dark."""
    img_np = np.array(img)
    inverted = 255 - img_np
    return Image.fromarray(inverted)

def apply_ring_colonies(size=(512, 512), count=50):
    """Draws hollow ring-shaped colonies."""
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    colonies = []

    for _ in range(count):
        x = random.randint(20, size[0] - 20)
        y = random.randint(20, size[1] - 20)
        r = random.randint(8, 14)
        stain = random.randint(100, 255)
        outer = (x - r, y - r, x + r, y + r)
        inner = (x - r//2, y - r//2, x + r//2, y + r//2)
        draw.ellipse(outer, fill=(0, 0, stain))
        draw.ellipse(inner, fill="white")
        colonies.append((x, y, r))

    return img, colonies

def apply_intensity_dropout(img):
    # Adds dark patches to simulate staining dropout
    draw = ImageDraw.Draw(img)
    for _ in range(10):
        x0 = random.randint(0, img.width - 50)
        y0 = random.randint(0, img.height - 50)
        x1 = x0 + random.randint(20, 80)
        y1 = y0 + random.randint(20, 80)
        draw.rectangle([x0, y0, x1, y1], fill=(220, 220, 220))  # light grey patch
    return img

# Distortion map
distortions = {
    "blur.png": apply_blur,
    "lighting_gradient.png": apply_lighting_gradient,
    "radial_glare.png": apply_radial_glare,
    "grid_overlay.png": apply_grid_overlay,
    "partial_occlusion.png": apply_partial_occlusion,
    "background_noise.png": apply_background_noise,
    "artifacts.png": apply_artifacts,
}

# Generate mdistorted images
for filename, distortion_fn in distortions.items():
    img, colonies = generate_colony_image()
    img = distortion_fn(img)
    img.save(f"synthetic_data/images/{filename}")
    with open(f"synthetic_data/colony_data/{filename.replace('.png', '.json')}", "w") as f:
        json.dump(colonies, f)
    save_mask(colonies, filename)

# Generate enhanced base_complex.png
generate_base_complex_image("base_complex.png")

# Generate base_clean.png (simple clean synthetic image with no distortion)
def generate_base_clean_image(filename="base_clean.png", size=(512, 512), count=50):
    img, colonies = generate_colony_image(size=size, count=count)
    img.save(f"synthetic_data/images/{filename}")
    with open(f"synthetic_data/colony_data/{filename.replace('.png', '.json')}", "w") as f:
        json.dump(colonies, f)
    save_mask(colonies, filename, size=size)

# EDGE CLIPPING
img, colonies = generate_colony_image()
img, modified_colonies = apply_edge_clipping(img, colonies)
img.save("synthetic_data/images/edge_clipping.png")
with open("synthetic_data/colony_data/edge_clipping.json", "w") as f:
    json.dump(modified_colonies, f)
save_mask(modified_colonies, "edge_clipping.png")

# INVERTED CONTRAST
img, colonies = generate_colony_image()
img = apply_inverted_contrast(img)
img.save("synthetic_data/images/inverted_contrast.png")
with open("synthetic_data/colony_data/inverted_contrast.json", "w") as f:
    json.dump(colonies, f)
save_mask(colonies, "inverted_contrast.png")

# RING COLONIES
img, colonies = apply_ring_colonies()
img.save("synthetic_data/images/ring_colonies.png")
with open("synthetic_data/colony_data/ring_colonies.json", "w") as f:
    json.dump(colonies, f)
save_mask(colonies, "ring_colonies.png")

# (Optional) Clean up any old files from previous runs so they don't confuse training / eval
def _safe_remove(path):
    try:
        if os.path.exists(path):
            os.remove(path)
    except Exception:
        pass

for base in ["intensity_dropout", "overlap"]:
    _safe_remove(f"synthetic_data/images/{base}.png")
    _safe_remove(f"synthetic_data/masks/{base}.png")
    _safe_remove(f"synthetic_data/colony_data/{base}.json")

# Special case: density gradient from left to right
def generate_density_gradient_image(filename="density_gradient.png", size=(1024, 512), columns=10):
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    colonies = []

    width, height = size
    col_width = width // columns
    base_density = 3  # Controls density increase per column

    for i in range(columns):
        x_min = i * col_width
        x_max = (i + 1) * col_width
        count = base_density * (i + 1) * 5  # Exponential-like increase

        for _ in range(count):
            x = random.randint(x_min + 5, x_max - 5)
            y = random.randint(10, height - 10)
            r = random.randint(3, 8)  # Smaller radius
            stain = random.randint(100, 255)
            draw.ellipse((x - r, y - r, x + r, y + r), fill=(0, 0, stain))
            colonies.append((x, y, r))

    img.save(f"synthetic_data/images/{filename}")
    with open(f"synthetic_data/colony_data/{filename.replace('.png', '.json')}", "w") as f:
        json.dump(colonies, f)
    save_mask(colonies, filename, size=size)

# Run density gradient generation
generate_density_gradient_image()

2. Display synthetic images

Creates a quick visual grid so generated synthetic examples can be inspected before processing.

In [ ]:
# Folder where your images are stored
image_folder = SYNTHETIC_IMAGE_DIR

# List of image filenames (manually or automatically detected)
filenames = sorted([
    f for f in os.listdir(image_folder)
    if f.endswith(".png")
])

# Grid display settings
n_cols = 3
n_rows = (len(filenames) + n_cols - 1) // n_cols
plt.figure(figsize=(15, 5 * n_rows))

# Loop through and show each image
for i, filename in enumerate(filenames):
    img_path = os.path.join(image_folder, filename)
    img = Image.open(img_path)

    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(img)
    plt.title(filename)
    plt.axis("off")

plt.tight_layout()
plt.show()

3. Optional resizing

Resizes synthetic images and masks. Images use bilinear interpolation; masks use nearest-neighbour interpolation to preserve binary labels.

In [ ]:
# Define paths
image_dir = SYNTHETIC_IMAGE_DIR
mask_dir = SYNTHETIC_MASK_DIR
resized_image_dir = PROJECT_ROOT / "synthetic_data" / "resized_images"
resized_mask_dir = PROJECT_ROOT / "synthetic_data" / "resized_masks"

# Create folders
os.makedirs(resized_image_dir, exist_ok=True)
os.makedirs(resized_mask_dir, exist_ok=True)

# Resize settings
target_size = (256, 256)

# Resize images
for filename in os.listdir(image_dir):
    if filename.endswith(".png"):
        img = Image.open(os.path.join(image_dir, filename))
        img = img.resize(target_size, Image.BILINEAR)  # smooth interpolation
        img.save(os.path.join(resized_image_dir, filename))

# Resize masks
for filename in os.listdir(mask_dir):
    if filename.endswith(".png"):
        mask = Image.open(os.path.join(mask_dir, filename))
        mask = mask.resize(target_size, Image.NEAREST)  # keep binary mask sharp
        mask.save(os.path.join(resized_mask_dir, filename))

4. Synthetic segmentation using the dual-mode classical detector

Runs the colony detector on synthetic images. Outputs predicted masks, overlays, and a CSV file containing predicted colony counts.

In [ ]:
# PATHS
# These variables now come from Section 0. Edit Section 0 only; do not hard-code local paths here
IMAGE_FOLDER = SYNTHETIC_IMAGE_DIR
OUT_DIR = SYNTHETIC_OUTPUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# KNOBS
MIN_AREA   = 25          # keep small colonies
MAX_AREA   = 5000        # cap very large merges
MIN_CIRC   = 0.12        # 0..1; lower = more permissive
USE_WATERSHED = True     # split overlaps for both modes
SIGMA_BG   = 31          # background blur sigma for lighting / glare (25 – 45 good)

# HELPER: component filter
def _filter_components(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros_like(mask)
    pts  = []
    for i in range(1, n):
        x,y,w,h,area = stats[i]
        if area < MIN_AREA or area > MAX_AREA:
            continue
        comp = (labels==i).astype(np.uint8)
        cnts,_ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        cnt = cnts[0]
        A = cv2.contourArea(cnt); P = cv2.arcLength(cnt, True) + 1e-6
        circ = 4*math.pi*A/(P*P)
        if circ < MIN_CIRC:
            continue
        keep[labels==i] = 255
        M = cv2.moments(cnt)
        cx = int(M["m10"]/(M["m00"]+1e-6)); cy = int(M["m01"]/(M["m00"]+1e-6))
        r  = int(max(3, round((A/np.pi)**0.5)))
        pts.append((cx,cy,r))
    return keep, pts

def _watershed_split(mask, guide_bgr, k=5, thr=0.35):
    if not USE_WATERSHED:
        return mask
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k))
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, thr*dist.max(), 255, 0); sure_fg = sure_fg.astype(np.uint8)
    sure_bg = cv2.dilate(mask, k5, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)
    n, markers = cv2.connectedComponents(sure_fg); markers = markers + 1
    markers[unknown==255] = 0
    ws = cv2.watershed(guide_bgr.copy(), markers)
    sep = np.zeros_like(mask)
    for lab in np.unique(ws):
        if lab <= 1: continue
        sep[ws==lab] = 255
    return sep

# CORE: dual‑mode detector
def detect_colonies(img_bgr):
    b,g,r = cv2.split(img_bgr)

    # Mode 1: blue‑contrast (your normal blue colonies)
    rgmax = cv2.max(r, g)
    diff  = cv2.subtract(b, rgmax)          # B - max(R,G), negatives clipped
    bg    = cv2.GaussianBlur(diff, (0,0), SIGMA_BG)
    flat  = cv2.subtract(diff, bg)
    flat  = cv2.normalize(flat, None, 0, 255, cv2.NORM_MINMAX)
    flat  = cv2.medianBlur(flat, 3)

    _, th_blue = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask_blue = cv2.morphologyEx(cv2.morphologyEx(th_blue, cv2.MORPH_OPEN, k3, 1),
                                 cv2.MORPH_CLOSE, k3, 1)
    mask_blue = _watershed_split(mask_blue, img_bgr, k=5, thr=0.35)
    mask1, pts1 = _filter_components(mask_blue)

    # If blue mode found a reasonable number, use it
    if len(pts1) >= 5:
        return mask1, pts1

    # Mode 2: bright‑saturated fallback (handles inverted_contrast)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H,S,V = cv2.split(hsv)
    V_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(V)
    _, th_v = cv2.threshold(V_eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Require some saturation so bright grey BG doesn't pass
    _, th_s = cv2.threshold(S, 40, 255, cv2.THRESH_BINARY)  # Tweak 25 – 60 as needed
    th_bs = cv2.bitwise_and(th_v, th_s)
    mask_fb = cv2.morphologyEx(cv2.morphologyEx(th_bs, cv2.MORPH_OPEN, k3, 1),
                               cv2.MORPH_CLOSE, k3, 1)
    mask_fb = _watershed_split(mask_fb, img_bgr, k=5, thr=0.35)
    mask2, pts2 = _filter_components(mask_fb)

    # If fallback still finds basically nothing, relax once (adaptive)
    if len(pts2) < 5:
        th_ad = cv2.adaptiveThreshold(V_eq, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 15, -3)
        th_ad = cv2.bitwise_and(th_ad, th_s)
        th_ad = cv2.morphologyEx(cv2.morphologyEx(th_ad, cv2.MORPH_OPEN, k3, 1),
                                 cv2.MORPH_CLOSE, k3, 1)
        th_ad = _watershed_split(th_ad, img_bgr, k=5, thr=0.35)
        mask3, pts3 = _filter_components(th_ad)
        if len(pts3) > len(pts2):
            return mask3, pts3

    return mask2, pts2

# Drawing 
def draw_overlay(img, points):
    over = img.copy()
    for (x,y,r) in points:
        cv2.circle(over, (int(x),int(y)), int(r), (0,255,0), 2)
    return over

# Run and show 
pngs = sorted(glob.glob(str(Path(IMAGE_FOLDER) / "*.png")))
if not pngs:
    raise SystemExit(f"No PNGs found in {IMAGE_FOLDER}")

csv_path = str(Path(OUT_DIR) / "counts_dualmode.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f); writer.writerow(["image","pred_count"])
    for p in pngs:
        img = cv2.imread(p)
        if img is None:
            print("Could not read", p);
            continue

        mask, pts = detect_colonies(img)
        overlay   = draw_overlay(img, pts)

        base = Path(p).stem
        cv2.imwrite(str(Path(OUT_DIR)/f"{base}_mask.png"), mask)
        cv2.imwrite(str(Path(OUT_DIR)/f"{base}_overlay.png"), overlay)
        writer.writerow([base, len(pts)])

        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(f"Original: {base}"); plt.axis('off')
        plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.title(f"Detections: {len(pts)}"); plt.axis('off')
        plt.show()

print(f"\nSaved outputs → {OUT_DIR}\nCSV → {csv_path}")

5. Synthetic overlay figure

Combines segmentation overlays into a publication-style grid.

In [ ]:
OUT_DIR = SYNTHETIC_OUTPUT_DIR
CSV_PATH = str(Path(OUT_DIR) / "counts_dualmode.csv")

label_map = {
    "base_clean": "Base (clean)",
    "base_complex": "Base (complex)",
    "artifacts": "Artifacts",
    "background_noise": "Background noise",
    "blur": "Blur",
    "concentric_rings": "Concentric rings",
    "edge_clipping": "Edge clipping",
    "inverted_contrast": "Inverted contrast",
    "lighting_gradient": "Lighting gradient",
    "partial_occlusion": "Partial occlusion",
    "radial_glare": "Radial glare",
    "ring_colonies": "Ring colonies",
    "grid_overlay": "Grid overlay",
    "density_gradient": "Density gradient",
}

# Read counts
counts = {}
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, newline="") as f:
        for row in csv.DictReader(f):
            counts[row["image"]] = int(row["pred_count"])

# Collect overlays 
paths = list(Path(OUT_DIR).glob("*_overlay.png"))
if not paths:
    raise SystemExit(f"No *_overlay.png files found in {OUT_DIR}")

# Reorder so last row = [item, item, density_gradient(span2)]
dg = [p for p in paths if p.stem.replace("_overlay","") == "density_gradient"]
others = [p for p in paths if p not in dg]
others.sort(key=lambda p: p.stem)  # Stable order for the rest
ordered = (others[:-2] + others[-2:] + dg) if len(others) >= 2 else (others + dg)

# layout knobs
cols         = 4
tile_w       = 520
outer_margin = 6
gutter       = 8
border_px    = 1

# BIGGER LABELS: scale these
FONT_SCALE   = 1.8   # try 1.6 – 2.2 depending on taste
BASE_TITLE   = 15
BASE_COUNT   = 15
P_X, P_Y     = int(6*FONT_SCALE), int(4*FONT_SCALE)  # Padding around label bg

def get_font(size=15):
    for fp in ["arial.ttf", r"C:\Windows\Fonts\arial.ttf"]:
        try:
            return ImageFont.truetype(fp, size)
        except:
            pass
    return ImageFont.load_default()

font_title = get_font(int(BASE_TITLE*FONT_SCALE))
font_count = get_font(int(BASE_COUNT*FONT_SCALE))

def span_for(stem: str) -> int:
    return 2 if stem == "density_gradient" else 1

# Preload, resize by span width 
items = []
for p in ordered:
    stem = p.stem.replace("_overlay","")
    img  = Image.open(p).convert("RGB")
    span = span_for(stem)
    target_w = tile_w*span + gutter*(span-1)
    w0, h0 = img.size
    target_h = int(h0 * (target_w / w0))  # Keep aspect by width
    items.append({"stem": stem, "img": img.resize((target_w, target_h), Image.LANCZOS), "span": span})

# Compute per-row heights with packing (left -> right wrap) 
row_heights, remaining, cur_h = [], cols, 0
for it in items:
    if it["span"] > remaining and remaining != cols:
        row_heights.append(cur_h); remaining, cur_h = cols, 0
    cur_h = max(cur_h, it["img"].size[1])
    remaining -= it["span"]
row_heights.append(cur_h)

W = outer_margin*2 + cols*tile_w + (cols-1)*gutter
H = outer_margin*2 + sum(row_heights) + (len(row_heights)-1)*gutter
fig  = Image.new("RGB", (W, H), "white")
draw = ImageDraw.Draw(fig)

# Compose 
y = outer_margin; remaining = cols; row_i = 0; x = outer_margin
for it in items:
    img, stem, span = it["img"], it["stem"], it["span"]
    iw, ih = img.size
    if span > remaining and remaining != cols:
        y += row_heights[row_i] + gutter
        row_i += 1
        x = outer_margin
        remaining = cols
    fig.paste(img, (x, y))

    # Border
    draw.rectangle([x, y, x+iw-1, y+ih-1], outline=(180,180,180), width=border_px)

    # Title (bigger font and bigger bg padding)
    title = label_map.get(stem, stem.replace("_"," ").capitalize())
    t_x1, t_y1, t_x2, t_y2 = draw.textbbox((0,0), title, font=font_title)
    t_w, t_h = t_x2 - t_x1, t_y2 - t_y1
    tx, ty = x + (iw - t_w)//2, y + int(6*FONT_SCALE)
    draw.rectangle([tx-P_X, ty-P_Y, tx+t_w+P_X, ty+t_h+P_Y], fill="white")
    draw.text((tx, ty), title, fill="black", font=font_title)

    # Count tag (bigger too)
    if stem in counts:
        tag = f"Count: {counts[stem]}"
        c_x1, c_y1, c_x2, c_y2 = draw.textbbox((0,0), tag, font=font_count)
        cw, ch = c_x2 - c_x1, c_y2 - c_y1
        cx, cy = x + iw - cw - int(10*FONT_SCALE), y + ih - ch - int(8*FONT_SCALE)
        draw.rectangle([cx-P_X, cy-P_Y, cx+cw+P_X, cy+ch+P_Y], fill="white")
        draw.text((cx, cy), tag, fill="black", font=font_count)

    x += iw + gutter
    remaining -= span

# Trim outer white frame 
bg = Image.new("RGB", fig.size, "white")
bbox = ImageChops.difference(fig, bg).getbbox()
if bbox:
    fig = fig.crop(bbox)

out_path = str(Path(OUT_DIR) / "synthetic_detection_overlays_grid_4col_SPAN2.png")
fig.save(out_path, dpi=(300,300))
print(f"Saved -> {out_path}")



6. Synthetic masks comparison panels

Compares original images, ground-truth masks, predicted masks, and error maps for visual validation.

In [ ]:
# SIDE-BY-SIDE: Original | GT mask | Pred mask | Error map
# FOLDERS
IMG_DIR  = SYNTHETIC_IMAGE_DIR
GT_DIR   = SYNTHETIC_MASK_DIR
PRED_DIR = SYNTHETIC_OUTPUT_DIR # Change to outputs_basic / outputs_classical to compare

# BEHAVIOUR
SHOW_INLINE = True     # Show each panel as it’s made
USE_OVERLAY = False    # True → show transparent error overlay on original instead of blocky map

# OUTPUTS
#  <PRED_DIR>/panels/<name>_panel.png
#  <PRED_DIR>/mask_comparison_summary.csv

import os, glob, csv
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as mpatches

IMG_DIR  = Path(IMG_DIR)
GT_DIR   = Path(GT_DIR)
PRED_DIR = Path(PRED_DIR)
PANELS   = PRED_DIR / "panels"
PANELS.mkdir(parents=True, exist_ok=True)

# Colours (very distinct)
PALETTE = {
    0: (0,   0,   0),   # Background
    1: (0, 255,   0),   # TP bright green
    2: (255, 0,   0),   # FP red
    3: (0, 102, 255),   # FN vivid blue
}

def read_mask(p):
    # Read a mask image -> binary 0 / 1 array (handles non-binary inputs}
    if not p.exists(): return None
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    _, m = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (m > 0).astype(np.uint8)

def dice_iou(gt, pr):
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    dice = (2*inter) / (gt.sum() + pr.sum() + 1e-8)
    iou  = inter / (union + 1e-8)
    return float(dice), float(iou)

def error_map_rgb(gt, pr):
    # Return RGB error map: 1=TP, 2=FP, 3=FN using PALETTE
    tp = np.logical_and(gt, pr)
    fp = np.logical_and(~gt.astype(bool), pr.astype(bool))
    fn = np.logical_and(gt.astype(bool), ~pr.astype(bool))
    em = np.zeros(gt.shape, np.uint8)
    em[tp] = 1; em[fp] = 2; em[fn] = 3
    h, w = gt.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for k, (r,g,b) in PALETTE.items():
        rgb[em == k] = (r,g,b)
    return rgb

def overlay_errors_on_image(img_rgb, gt, pr, alpha=0.35):
    # Transparent colour overlay on the original image for TP/FP/FN
    mask = np.zeros_like(img_rgb, dtype=np.uint8)
    tp = np.logical_and(gt, pr)
    fp = np.logical_and(~gt.astype(bool), pr.astype(bool))
    fn = np.logical_and(gt.astype(bool), ~pr.astype(bool))
    mask[tp] = PALETTE[1]
    mask[fp] = PALETTE[2]
    mask[fn] = PALETTE[3]
    return cv2.addWeighted(img_rgb, 1.0, mask, alpha, 0)

def make_panel(name):
    img_p = IMG_DIR / f"{name}.png"
    gt_p  = GT_DIR  / f"{name}.png"
    pr_p  = PRED_DIR / f"{name}_mask.png"   # Detector output naming

    if not (img_p.exists() and gt_p.exists() and pr_p.exists()):
        print(f"[skip] missing file(s) for {name}")
        return None

    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    gt  = read_mask(gt_p)
    pr  = read_mask(pr_p)
    if gt is None or pr is None or gt.shape != pr.shape:
        print(f"[skip] bad mask(s) for {name}")
        return None

    d, i = dice_iou(gt, pr)
    err_block = error_map_rgb(gt, pr)
    err_overlay = overlay_errors_on_image(img, gt, pr)

    # Figure
    plt.figure(figsize=(12,4))
    plt.suptitle(f"{name}   Dice={d:.3f}  IoU={i:.3f}", y=1.02, fontsize=12)

    plt.subplot(1,4,1); plt.imshow(img); plt.title("Original"); plt.axis("off")
    plt.subplot(1,4,2); plt.imshow(gt, cmap="gray"); plt.title("GT mask"); plt.axis("off")
    plt.subplot(1,4,3); plt.imshow(pr, cmap="gray"); plt.title("Pred mask"); plt.axis("off")

    if USE_OVERLAY:
        plt.subplot(1,4,4); plt.imshow(err_overlay); plt.title("Error overlay"); plt.axis("off")
    else:
        plt.subplot(1,4,4); plt.imshow(err_block);   plt.title("Error map");   plt.axis("off")

    # Legend
    legend_handles = [
        mpatches.Patch(color=np.array(PALETTE[1])/255.0, label="TP"),
        mpatches.Patch(color=np.array(PALETTE[2])/255.0, label="FP"),
        mpatches.Patch(color=np.array(PALETTE[3])/255.0, label="FN"),
    ]
    plt.legend(handles=legend_handles, loc="lower right", frameon=True, fontsize=9)

    plt.tight_layout()
    out = PANELS / f"{name}_panel.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    if SHOW_INLINE:
        plt.show()
    plt.close()
    return d, i, out

# Run for every predicted mask we have 
names = sorted({Path(p).stem.replace("_mask","") for p in glob.glob(str(PRED_DIR / "*_mask.png"))})

summary = []
for n in names:
    res = make_panel(n)
    if res:
        d, i, out = res
        summary.append({"image": n, "dice": f"{d:.4f}", "iou": f"{i:.4f}", "panel_path": str(out)})

# Save small CSV summary
csv_path = PRED_DIR / "mask_comparison_summary.csv"
if summary:
    with open(csv_path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=["image","dice","iou","panel_path"])
        wr.writeheader(); wr.writerows(summary)
    print(f"Saved {len(summary)} panels → {PANELS}")
    print(f"Summary CSV → {csv_path}")
else:
    print("No panels created. Check folder paths and that *_mask.png exist in your PRED_DIR.")

# Optional: show all saved panels in a grid for a quick scan
panel_paths = sorted(glob.glob(str(PANELS / "*.png")))
if panel_paths:
    cols = 3
    rows = int(np.ceil(len(panel_paths) / cols))
    plt.figure(figsize=(14, 4*rows))
    for idx, p in enumerate(panel_paths, 1):
        img = np.array(Image.open(p))
        plt.subplot(rows, cols, idx)
        plt.imshow(img); plt.axis('off')
        plt.title(Path(p).name, fontsize=9)
    plt.tight_layout()
    plt.show()

7. Synthetic segmentation metrics

Aggregates Dice, IoU, precision, and recall by synthetic image condition.

In [ ]:
GT_DIR       = SYNTHETIC_MASK_DIR
PRED_DIR     = SYNTHETIC_OUTPUT_DIR
per_img_csv  = PRED_DIR / "mask_comparison_summary.csv"  
out_csv      = PRED_DIR / "synthetic_metrics_by_condition.csv"

# load per-image metrics
df = pd.read_csv(per_img_csv)  # Columns: image, dice, iou, panel_path
df["dice"] = df["dice"].astype(float)
df["iou"]  = df["iou"].astype(float)

# Compute precision/recall per image from masks (robust to size mismatch)
def read_mask_bin(p: Path):
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    _, b = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (b > 0)

def prec_recall_for(stem: str):
    gt_p = GT_DIR  / f"{stem}.png"
    pr_p = PRED_DIR / f"{stem}_mask.png"
    GT = read_mask_bin(gt_p)
    PR = read_mask_bin(pr_p)
    if GT is None or PR is None:
        return np.nan, np.nan
    if GT.shape != PR.shape:
        PR = cv2.resize(PR.astype(np.uint8), (GT.shape[1], GT.shape[0]),
                        interpolation=cv2.INTER_NEAREST).astype(bool)
    GTb, PRb = GT.astype(bool), PR.astype(bool)
    TP = np.logical_and(GTb, PRb).sum()
    FP = np.logical_and(~GTb, PRb).sum()
    FN = np.logical_and(GTb, ~PRb).sum()
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    recall    = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return precision, recall

df["stem"] = df["image"].str.replace(r"\.png$", "", regex=True)
df[["precision","recall"]] = df["stem"].apply(lambda s: pd.Series(prec_recall_for(s)))

# Filename → condition mapping
COND_ORDER = [
    "Base (clean)", "Base (complex)", "Artifacts", "Background noise", "Blur",
    "Concentric rings", "Edge clipping", "Grid overlay", "Inverted contrast",
    "Lighting gradient", "Partial occlusion", "Radial glare", "Ring colonies",
    "Density gradient",
]
KEYWORDS = [
    (r"(?:^|_)base(?:_)?clean(?:_|$)|(?:^|_)clean(?:_|$)",                     "Base (clean)"),
    (r"(?:^|_)base(?:_)?complex(?:_|$)|(?:^|_)complex(?:_|$)",                 "Base (complex)"),
    (r"(?:^|_)(?:artifact|artefact)s?(?:_|$)",                                 "Artifacts"),
    (r"(?:^|_)(?:background_)?noise(?:_|$)",                                   "Background noise"),
    (r"(?:^|_)(?:blur|gaussian)(?:_|$)",                                       "Blur"),
    (r"(?:^|_)(?:concentric(?:_)?rings?|annulus)(?:_|$)",                      "Concentric rings"),
    (r"(?:^|_)(?:edge(?:_)?clipp?ing|clip)(?:_|$)",                            "Edge clipping"),
    (r"(?:^|_)grid(?:_)?overlay?(?:_|$)|(?:^|_)grid(?:_|$)",                   "Grid overlay"),
    (r"(?:^|_)(?:invert(?:ed)?|neg(?:ative)?)(?:_|$)",                         "Inverted contrast"),
    (r"(?:^|_)(?:lighting|illumination|illum)(?:_)?gradient(?:_|$)",           "Lighting gradient"),
    (r"(?:^|_)(?:partial(?:_)?occlusion|occlud(?:e|er|ed|ing)?)(?:_|$)",       "Partial occlusion"),
    (r"(?:^|_)(?:radial(?:_)?glare|glare)(?:_|$)",                             "Radial glare"),
    (r"(?:^|_)ring(?:_)?colon(?:y|ies)(?:_|$)",                                "Ring colonies"),
    (r"(?:^|_)density(?:_)?gradient(?:_|$)|(?:^|_)crowd(?:ed|ing)?(?:_|$)",    "Density gradient"),
]
def to_condition(name: str) -> str:
    base = name.lower().replace("__","_").replace("-","_").replace(" ","_")
    for pat, label in KEYWORDS:
        if re.search(pat, base):
            return label
    if base.startswith("base"): return "Base (clean)"
    return "Artifacts"

df["condition"] = df["image"].map(to_condition)

# Aggregate means and SDs
agg = df.groupby("condition").agg(
    dice_mean=("dice", np.mean), dice_sd=("dice", np.std),
    iou_mean=("iou", np.mean),   iou_sd=("iou", np.std),
    precision_mean=("precision", np.mean), precision_sd=("precision", np.std),
    recall_mean=("recall", np.mean),       recall_sd=("recall", np.std),
    n=("image","count")
).reset_index()

# Order rows like the paper
cat = pd.Categorical(agg["condition"], categories=COND_ORDER, ordered=True)
agg = agg.assign(condition=cat).sort_values("condition").dropna(subset=["condition"])

# Save CSV 
agg.to_csv(out_csv, index=False)
print(f"[OK] Wrote per-condition metrics → {out_csv}")

# Print LaTeX rows (Condition, Dice, IoU, Precision, Recall)
def fmt(m, s):
    return f"{m:.3f} $\\pm$ {(0 if pd.isna(s) else s):.3f}"

print("\n% ---- Paste these rows into your table ----")
for _, r in agg.iterrows():
    print(
        f"{r['condition']} & "
        f"{fmt(r['dice_mean'], r['dice_sd'])} & "
        f"{fmt(r['iou_mean'], r['iou_sd'])} & "
        f"{fmt(r['precision_mean'], r['precision_sd'])} & "
        f"{fmt(r['recall_mean'], r['recall_sd'])} \\\\"
    )

8. Synthetic error-map figure

Builds a compact multi-condition figure showing true positives, false positives, and false negatives.

In [ ]:
GT_DIR   = SYNTHETIC_MASK_DIR
PRED_DIR = SYNTHETIC_OUTPUT_DIR
OUT_PATH = PRED_DIR / "error_maps_grid_span2.png"

# Display names and order
label_map = {
    "artifacts": "Artifacts",
    "background_noise": "Background noise",
    "base_clean": "Base (clean)",
    "base_complex": "Base (complex)",
    "blur": "Blur",
    "concentric_rings": "Concentric rings",
    "edge_clipping": "Edge clipping",
    "grid_overlay": "Grid overlay",
    "inverted_contrast": "Inverted contrast",
    "lighting_gradient": "Lighting gradient",
    "partial_occlusion": "Partial occlusion",
    "radial_glare": "Radial glare",
    "ring_colonies": "Ring colonies",
    "density_gradient": "Density gradient",   # Will span 2 columns
}
order = list(label_map.keys())

# Colors (match your example)
COL_TP = (0, 255, 0)     # green
COL_FP = (255, 0, 0)     # red
COL_FN = (0, 102, 255)   # blue
BG     = (0, 0, 0)       # black background

def get_font(size=16):
    for fp in ["arial.ttf", r"C:\Windows\Fonts\arial.ttf"]:
        try:
            return ImageFont.truetype(fp, size)
        except:
            pass
    return ImageFont.load_default()

def read_mask_bin(p: Path):
    # Read PNG -> binary mask (bool). Works with grayscale or 0 / 255
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    _, b = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (b > 0)

# BIGGER LABELS: scale these
FONT_SCALE = 1.8                 # try 1.6 – 2.2
BASE_TITLE = 15
P_X = int(6 * FONT_SCALE)        # horizontal padding of white bg
P_Y = int(4 * FONT_SCALE)        # vertical padding of white bg
TITLE_Y_OFFSET = int(6 * FONT_SCALE)

font_title = get_font(int(BASE_TITLE * FONT_SCALE))

# Build per-condition error maps (from GT vs Pred)
items = []   # List of dicts: {"img": PIL.Image, "stem": str, "span": int}
for stem in order:
    gt_p  = GT_DIR  / f"{stem}.png"
    pr_p  = PRED_DIR / f"{stem}_mask.png"
    if not (gt_p.exists() and pr_p.exists()):
        continue

    GT = read_mask_bin(gt_p)
    PR = read_mask_bin(pr_p)
    if GT is None or PR is None:
        continue
    if GT.shape != PR.shape:
        PR = cv2.resize(PR.astype(np.uint8), (GT.shape[1], GT.shape[0]),
                        interpolation=cv2.INTER_NEAREST).astype(bool)

    GTb, PRb = GT.astype(bool), PR.astype(bool)

    # True error map on black background
    h, w = GTb.shape
    err = np.zeros((h, w, 3), dtype=np.uint8); err[:] = BG
    TP = np.logical_and(GTb, PRb)
    FP = np.logical_and(~GTb, PRb)
    FN = np.logical_and(GTb, ~PRb)
    err[TP] = COL_TP; err[FP] = COL_FP; err[FN] = COL_FN

    im = Image.fromarray(err)
    span = 2 if stem == "density_gradient" else 1
    items.append({"img": im, "stem": stem, "span": span})

# Lay out as a 4-column grid; density spans 2 columns
if not items:
    raise SystemExit("No matching GT/pred masks found.")

cols         = 4
tile_w       = 520      # width of one column; density gets 2*tile_w + gutter
gutter       = 8
margin       = 8
border_px    = 1

# Resize by span width (preserve aspect)
for it in items:
    im, span = it["img"], it["span"]
    target_w = tile_w*span + gutter*(span-1)
    w, h = im.size
    new_h = int(h * (target_w / w))
    it["img"] = im.resize((target_w, new_h), Image.LANCZOS)

# Compute row heights by packing left to right
row_heights, remaining, cur_h = [], cols, 0
for it in items:
    if it["span"] > remaining and remaining != cols:
        row_heights.append(cur_h); remaining, cur_h = cols, 0
    cur_h = max(cur_h, it["img"].size[1])
    remaining -= it["span"]
row_heights.append(cur_h)

W = margin*2 + cols*tile_w + (cols-1)*gutter
H = margin*2 + sum(row_heights) + (len(row_heights)-1)*gutter
fig  = Image.new("RGB", (W, H), "white")
draw = ImageDraw.Draw(fig)

# Paste panels and titles
y = margin; remaining = cols; row_i = 0; x = margin
for it in items:
    im, stem, span = it["img"], it["stem"], it["span"]
    iw, ih = im.size
    if span > remaining and remaining != cols:
        y += row_heights[row_i] + gutter
        row_i += 1
        x = margin
        remaining = cols

    fig.paste(im, (x, y))
    draw.rectangle([x, y, x+iw-1, y+ih-1], outline=(180,180,180), width=border_px)

    title = label_map[stem]
    x1,y1,x2,y2 = draw.textbbox((0,0), title, font=font_title)
    t_w, t_h = x2-x1, y2-y1
    tx, ty = x + (iw - t_w)//2, y + TITLE_Y_OFFSET
    draw.rectangle([tx-P_X, ty-P_Y, tx+t_w+P_X, ty+t_h+P_Y], fill="white")
    draw.text((tx, ty), title, fill="black", font=font_title)

    x += iw + gutter
    remaining -= span

# Trim outer white frame and save (300 dpi)
bg = Image.new("RGB", fig.size, "white")
bbox = ImageChops.difference(fig, bg).getbbox()
if bbox: fig = fig.crop(bbox)
fig.save(OUT_PATH, dpi=(300,300))
print(f"Saved → {OUT_PATH}")

9. Real experimental image segmentation

Runs the same dual-mode detector on real clonogenic assay images. Manual masks are not required for this step.

In [ ]:
# Folders
# These variables now come from Section 0. Edit Section 0 only; do not hard-code local paths here
REAL_IMG_DIR = REAL_IMAGE_DIR
OUT_DIR = REAL_OUTPUT_DIR

os.makedirs(OUT_DIR, exist_ok=True)

# Detection knobs (you can tweak)
MIN_AREA   = 25
MAX_AREA   = 5000
MIN_CIRC   = 0.12
USE_WATERSHED = True
SIGMA_BG   = 31   # Background blur sigma (25 – 45 good)

# Helpers (same as before)
def _filter_components(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros_like(mask)
    pts  = []
    for i in range(1, n):
        x,y,w,h,area = stats[i]
        if area < MIN_AREA or area > MAX_AREA:
            continue
        comp = (labels==i).astype(np.uint8)
        cnts,_ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        cnt = cnts[0]
        A = cv2.contourArea(cnt); P = cv2.arcLength(cnt, True) + 1e-6
        circ = 4*math.pi*A/(P*P)
        if circ < MIN_CIRC:
            continue
        keep[labels==i] = 255
        M = cv2.moments(cnt)
        cx = int(M["m10"]/(M["m00"]+1e-6)); cy = int(M["m01"]/(M["m00"]+1e-6))
        r  = int(max(3, round((A/np.pi)**0.5)))
        pts.append((cx,cy,r))
    return keep, pts

def _watershed_split(mask, guide_bgr, k=5, thr=0.35):
    if not USE_WATERSHED:
        return mask
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k))
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, thr*dist.max(), 255, 0); sure_fg = sure_fg.astype(np.uint8)
    sure_bg = cv2.dilate(mask, k5, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)
    n, markers = cv2.connectedComponents(sure_fg); markers = markers + 1
    markers[unknown==255] = 0
    ws = cv2.watershed(guide_bgr.copy(), markers)
    sep = np.zeros_like(mask)
    for lab in np.unique(ws):
        if lab <= 1: continue
        sep[ws==lab] = 255
    return sep

def detect_colonies(img_bgr):
    # Dual-mode detector: blue-contrast first, HSV fallback for odd contrast
    b,g,r = cv2.split(img_bgr)

    # Mode 1: blue-contrast (typical blue-stained colonies)
    rgmax = cv2.max(r, g)
    diff  = cv2.subtract(b, rgmax)          # B - max(R,G)
    bg    = cv2.GaussianBlur(diff, (0,0), SIGMA_BG)
    flat  = cv2.subtract(diff, bg)
    flat  = cv2.normalize(flat, None, 0, 255, cv2.NORM_MINMAX)
    flat  = cv2.medianBlur(flat, 3)

    _, th_blue = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask_blue = cv2.morphologyEx(cv2.morphologyEx(th_blue, cv2.MORPH_OPEN, k3, 1),
                                 cv2.MORPH_CLOSE, k3, 1)
    mask_blue = _watershed_split(mask_blue, img_bgr, k=5, thr=0.35)
    mask1, pts1 = _filter_components(mask_blue)
    if len(pts1) >= 5:
        return mask1, pts1

    # Mode 2: HSV fallback (handles inverted / odd lighting)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H,S,V = cv2.split(hsv)
    V_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(V)
    _, th_v = cv2.threshold(V_eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, th_s = cv2.threshold(S, 40, 255, cv2.THRESH_BINARY)
    th_bs = cv2.bitwise_and(th_v, th_s)
    mask_fb = cv2.morphologyEx(cv2.morphologyEx(th_bs, cv2.MORPH_OPEN, k3, 1),
                               cv2.MORPH_CLOSE, k3, 1)
    mask_fb = _watershed_split(mask_fb, img_bgr, k=5, thr=0.35)
    mask2, pts2 = _filter_components(mask_fb)

    # One relaxed pass if needed
    if len(pts2) < 5:
        th_ad = cv2.adaptiveThreshold(V_eq, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 15, -3)
        th_ad = cv2.bitwise_and(th_ad, th_s)
        th_ad = cv2.morphologyEx(cv2.morphologyEx(th_ad, cv2.MORPH_OPEN, k3, 1),
                                 cv2.MORPH_CLOSE, k3, 1)
        th_ad = _watershed_split(th_ad, img_bgr, k=5, thr=0.35)
        mask3, pts3 = _filter_components(th_ad)
        if len(pts3) > len(pts2):
            return mask3, pts3

    return mask2, pts2

def draw_overlay(img_bgr, points):
    over = img_bgr.copy()
    for (x,y,r) in points:
        cv2.circle(over, (int(x),int(y)), int(r), (0,255,0), 2)
    return over

def overlay_mask_on_image(img_rgb, mask_bin, color=(0,255,0), alpha=0.35):
    # Transparent overlay of predicted mask on original image
    col = np.zeros_like(img_rgb, dtype=np.uint8)
    col[mask_bin>0] = color
    return cv2.addWeighted(img_rgb, 1.0, col, alpha, 0)

# Run on all real images
extensions = ("*.png","*.jpg","*.jpeg","*.tif","*.tiff","*.bmp")
files = []
for ext in extensions:
    files.extend(glob.glob(str(Path(REAL_IMG_DIR)/ext)))
files = sorted(files)
if not files:
    raise SystemExit(f"No images found in {REAL_IMG_DIR}")

for p in files:
    img_bgr = cv2.imread(p)
    if img_bgr is None:
        print("Could not read", p)
        continue

    mask, pts = detect_colonies(img_bgr)

    base = Path(p).stem
    # Save binary mask
    cv2.imwrite(str(Path(OUT_DIR)/f"{base}_mask.png"), mask)

    # Save circle overlay (green outlines)
    overlay_circles = draw_overlay(img_bgr, pts)
    cv2.imwrite(str(Path(OUT_DIR)/f"{base}_overlay_circles.png"), overlay_circles)

    # Save transparent area overlay on original
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    overlay_area = overlay_mask_on_image(img_rgb, mask>0, color=(0,255,0), alpha=0.35)
    overlay_area_bgr = cv2.cvtColor(overlay_area, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(Path(OUT_DIR)/f"{base}_overlay.png"), overlay_area_bgr)

    # Show quick triptych: Original | Pred mask | Overlay
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img_rgb); plt.title(f"Original: {base}"); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(mask, cmap='gray'); plt.title(f"Pred mask ({len(pts)} colonies)"); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(overlay_area); plt.title("Overlay"); plt.axis('off')
    plt.tight_layout(); plt.show()

print(f"\nSaved outputs → {OUT_DIR}")

10. Real-image visual comparison against manual masks

Uses manual annotation masks, if available, to create original / ground-truth / prediction / error-overlay panels.

In [ ]:
# Tight grid without legend 
REAL_IMG_DIR = REAL_IMAGE_DIR
GT_MASK_DIR = REAL_MASK_DIR
OUT_DIR = REAL_OUTPUT_DIR

def resolve_with_any_ext(folder, stem):
    for ext in (".png",".jpg",".jpeg",".tif",".tiff",".bmp"):
        p = Path(folder)/f"{stem}{ext}"
        if p.exists(): return str(p)
    return None

def load_gray_mask(folder, stem):
    p = resolve_with_any_ext(folder, stem)
    m = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if m.ndim==3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m>0).astype(np.uint8)

def compute_error_overlay(img_rgb, pred_bin, gt_bin, alpha=0.35):
    pred = pred_bin.astype(bool); gt = gt_bin.astype(bool)
    TP = np.logical_and(pred, gt)
    FP = np.logical_and(pred, np.logical_not(gt))
    FN = np.logical_and(np.logical_not(pred), gt)
    err = np.zeros_like(img_rgb)
    err[TP] = (0,255,0)
    err[FP] = (255,0,0)
    err[FN] = (255,255,0)
    return cv2.addWeighted(img_rgb, 1.0, err, alpha, 0)

# Automatically pair each real image with its manual mask
# A third-party user only needs to name masks as image_stem and one of: _mask, _manual, _binary, _gt, _groundtruth, or _ground_truth
rows = matched_real_image_mask_rows(REAL_IMG_DIR, GT_MASK_DIR)
if not rows:
    raise FileNotFoundError(
        f"No matching image / mask pairs found. Check REAL_IMAGE_DIR={REAL_IMG_DIR} and REAL_MASK_DIR={GT_MASK_DIR}."
    )
headers = ["Original image", "Manual ground truth", "Automated segmentation", "Error overlay"]

plt.rcParams.update({"font.size":9})

fig, axes = plt.subplots(len(rows), len(headers),
                         figsize=(7.2, 6.0))  # Smaller height to reduce white space

for j,h in enumerate(headers):
    axes[0,j].set_title(h, pad=2)  # Pad=2 pulls titles closer

for i,(img_stem, gt_stem) in enumerate(rows):
    img_path = resolve_with_any_ext(REAL_IMG_DIR, img_stem)
    img_bgr = cv2.imread(img_path); img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gt_bin = load_gray_mask(GT_MASK_DIR, gt_stem)
    gt_bin = cv2.resize(gt_bin,(img_rgb.shape[1],img_rgb.shape[0]),interpolation=cv2.INTER_NEAREST)
    pred_mask_path = Path(OUT_DIR)/f"{img_stem}_mask.png"
    pred_bin = cv2.imread(str(pred_mask_path), cv2.IMREAD_GRAYSCALE)
    pred_bin = (pred_bin>0).astype(np.uint8)
    err_overlay = compute_error_overlay(img_rgb,pred_bin,gt_bin)

    panels=[img_rgb, gt_bin, pred_bin, err_overlay]
    cmaps=[None,"gray","gray",None]
    for j in range(4):
        ax=axes[i,j]
        if cmaps[j]: ax.imshow(panels[j], cmap=cmaps[j], vmin=0, vmax=1)
        else: ax.imshow(panels[j])
        ax.axis("off")

plt.tight_layout(pad=0.2, w_pad=0.2, h_pad=0.2)  # Tighter padding between subplots
out = Path(OUT_DIR)/"real_1_2_4_tight.png"
plt.savefig(out, dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.close()
print("Saved tight figure →", out)


11. Real Dice and IoU metrics

Computes quantitative agreement between predicted masks and manual annotation masks.

In [ ]:
REAL_IMG_DIR = REAL_IMAGE_DIR
GT_MASK_DIR = REAL_MASK_DIR
OUT_DIR = REAL_OUTPUT_DIR

rows = matched_real_image_mask_rows(REAL_IMG_DIR, GT_MASK_DIR)
if not rows:
    raise FileNotFoundError(
        f"No matching image / mask pairs found. Check REAL_IMAGE_DIR={REAL_IMG_DIR} and REAL_MASK_DIR={GT_MASK_DIR}."
    )

def resolve_with_any_ext(folder, stem):
    for ext in (".png",".jpg",".jpeg",".tif",".tiff",".bmp"):
        p = Path(folder)/f"{stem}{ext}"
        if p.exists(): return str(p)
    return None

def load_binary_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None: raise IOError(f"Cannot read {path}")
    if m.ndim==3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m>0).astype(np.uint8)

def dice_iou(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    a = pred.sum(); b = gt.sum()
    dice = (2*inter)/(a+b) if (a+b)>0 else np.nan
    iou  = inter/np.logical_or(pred,gt).sum() if np.logical_or(pred,gt).sum()>0 else np.nan
    return dice, iou, inter, a, b

metrics_csv = Path(OUT_DIR)/"real_image_metrics.csv"
with open(metrics_csv,"w",newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Image","GT_pixels","Pred_pixels","TP_pixels","Dice","IoU"])
    all_dice, all_iou = [], []
    for img_stem, gt_stem in rows:
        gt_path   = resolve_with_any_ext(GT_MASK_DIR, gt_stem)
        pred_path = Path(OUT_DIR)/f"{img_stem}_mask.png"
        gt_bin    = load_binary_mask(gt_path)
        pred_bin  = load_binary_mask(pred_path)
        if gt_bin.shape!=pred_bin.shape:
            pred_bin = cv2.resize(pred_bin,(gt_bin.shape[1],gt_bin.shape[0]),interpolation=cv2.INTER_NEAREST)
        dice, iou, tp, a, b = dice_iou(pred_bin,gt_bin)
        all_dice.append(dice); all_iou.append(iou)
        writer.writerow([img_stem,b,a,tp,f"{dice:.4f}",f"{iou:.4f}"])
    writer.writerow(["Mean ± SD","","","",
                     f"{np.mean(all_dice):.4f} ± {np.std(all_dice,ddof=1):.4f}",
                     f"{np.mean(all_iou):.4f} ± {np.std(all_iou,ddof=1):.4f}"])

print(f"Saved metrics → {metrics_csv}")



12. Real ROC / AUC analysis

Computes ROC curves and AUC from the pre-threshold blue-contrast feature map using manual masks as ground truth.

In [ ]:
REAL_IMG_DIR = REAL_IMAGE_DIR
GT_MASK_DIR = REAL_MASK_DIR
OUT_DIR = REAL_OUTPUT_DIR

rows = matched_real_image_mask_rows(REAL_IMG_DIR, GT_MASK_DIR)
if not rows:
    raise FileNotFoundError(
        f"No matching image / mask pairs found. Check REAL_IMAGE_DIR={REAL_IMG_DIR} and REAL_MASK_DIR={GT_MASK_DIR}."
    )

# Detector knobs (match your pipeline)
SIGMA_BG = 31     # Gaussian sigma used for background removal
MED_K    = 3      # Median blur on flat map (keeps noise down but preserves contrast)

# Helpers
def resolve_with_any_ext(folder, stem):
    for ext in (".png",".jpg",".jpeg",".tif",".tiff",".bmp"):
        p = Path(folder) / f"{stem}{ext}"
        if p.exists(): return str(p)
    return None

def load_rgb(folder, stem):
    p = resolve_with_any_ext(folder, stem)
    if p is None: raise FileNotFoundError(f"Image '{stem}.*' not found in {folder}")
    bgr = cv2.imread(p)
    if bgr is None: raise IOError(f"Failed to read {p}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), p

def load_binary_mask(folder, stem):
    p = resolve_with_any_ext(folder, stem)
    if p is None: raise FileNotFoundError(f"Mask '{stem}.*' not found in {folder}")
    m  = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if m is None: raise IOError(f"Failed to read {p}")
    if m.ndim==3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m>0).astype(np.uint8), p

def compute_flat_map(img_rgb, sigma_bg=31, med_k=3):
    # Reproduce the contrast map used before Otsu: (B - max(R,G)) minus Gaussian background, normalized [0,255]
    b, g, r = cv2.split(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    rgmax   = cv2.max(r, g)
    diff    = cv2.subtract(b, rgmax).astype(np.int16)         # Can go negative
    bg      = cv2.GaussianBlur(diff, (0,0), sigma_bg)
    flat    = cv2.subtract(diff, bg)                          # High where colony-like
    # Normalise to 8-bit
    flat    = flat.astype(np.float32)
    mn, mx  = np.min(flat), np.max(flat)
    if mx - mn < 1e-6:
        flat_n = np.zeros_like(flat, dtype=np.uint8)
    else:
        flat_n = ((flat - mn) * (255.0/(mx - mn))).astype(np.uint8)
    if med_k and med_k>1:
        flat_n = cv2.medianBlur(flat_n, med_k)
    return flat_n  # uint8 (0..255), higher = more colony-like

def confusion_from_thresh(score_u8, gt_bin, thr):
    # Threshold score map at 'thr' (>= thr = positive). Returns TP, FP, TN, FN counts
    pred = (score_u8 >= thr)
    gt   = (gt_bin > 0)
    TP = np.logical_and(pred, gt).sum(dtype=np.int64)
    FP = np.logical_and(pred, ~gt).sum(dtype=np.int64)
    TN = np.logical_and(~pred, ~gt).sum(dtype=np.int64)
    FN = np.logical_and(~pred, gt).sum(dtype=np.int64)
    return TP, FP, TN, FN

def metrics_from_confusion(TP, FP, TN, FN):
    tpr = TP / (TP + FN) if (TP+FN)>0 else np.nan                 # Recall
    fpr = FP / (FP + TN) if (FP+TN)>0 else np.nan
    prec = TP / (TP + FP) if (TP+FP)>0 else np.nan
    dice = 2*TP / (2*TP + FP + FN) if (2*TP+FP+FN)>0 else np.nan  # F1
    iou  = TP / (TP + FP + FN) if (TP+FP+FN)>0 else np.nan
    return tpr, fpr, prec, dice, iou

def auc_trapz(fpr, tpr):
    # Area under ROC via trapezoidal rule; sorts by FPR first
    fpr = np.asarray(fpr); tpr = np.asarray(tpr)
    idx = np.argsort(fpr)
    return np.trapz(tpr[idx], fpr[idx])

# Compute ROC for each image
OUT_DIR = Path(OUT_DIR); OUT_DIR.mkdir(parents=True, exist_ok=True)
aucs = []
plt.figure(figsize=(6.2, 5.0))
for img_stem, gt_stem in rows:
    # Load data
    img_rgb, _ = load_rgb(REAL_IMG_DIR, img_stem)
    gt_bin, _  = load_binary_mask(GT_MASK_DIR, gt_stem)
    H, W = img_rgb.shape[:2]
    if gt_bin.shape != (H, W):
        gt_bin = cv2.resize(gt_bin, (W, H), interpolation=cv2.INTER_NEAREST)

    # Score map (pre-threshold 'flat')
    flat = compute_flat_map(img_rgb, sigma_bg=SIGMA_BG, med_k=MED_K)

    # Sweep thresholds 0..255
    TPR, FPR, PRE, DICE, IOU, THR = [], [], [], [], [], []
    for thr in range(0, 256):
        TP, FP, TN, FN = confusion_from_thresh(flat, gt_bin, thr)
        tpr, fpr, pre, dice, iou = metrics_from_confusion(TP, FP, TN, FN)
        if np.isnan(tpr) or np.isnan(fpr):  # Skip degenerate (shouldn't usually happen)
            continue
        TPR.append(tpr); FPR.append(fpr); PRE.append(pre); DICE.append(dice); IOU.append(iou); THR.append(thr)

    # Ensure endpoints present (0,0) and (1,1)
    TPR = np.array(TPR); FPR = np.array(FPR)
    # Compute AUC
    auc = auc_trapz(FPR, TPR)
    aucs.append(auc)

    # Save per-image sweep CSV
    csv_path = OUT_DIR / f"{img_stem}_roc_sweep.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["threshold","TPR","FPR","Precision","F1(Dice)","IoU"])
        for k in range(len(THR)):
            w.writerow([THR[k], f"{TPR[k]:.6f}", f"{FPR[k]:.6f}", f"{PRE[k]:.6f}", f"{DICE[k]:.6f}", f"{IOU[k]:.6f}"])
    print(f"Saved ROC sweep → {csv_path}  |  AUC={auc:.4f}")

    # Plot ROC for this image
    idx = np.argsort(FPR)
    plt.plot(np.array(FPR)[idx], np.array(TPR)[idx], lw=2, label=f"{img_stem} (AUC={auc:.3f})")

# Macro-average AUC
macro_auc = float(np.mean(aucs)) if aucs else float("nan")

# Finalise ROC plot
plt.plot([0,1],[0,1], ls="--", lw=1, color="grey")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.xlabel("False Positive Rate (1 - specificity)")
plt.ylabel("True Positive Rate (recall)")
plt.title(f"Pixel-wise ROC (pre-threshold score sweep)\nMacro AUC = {macro_auc:.3f}")
plt.legend(loc="lower right", fontsize=8)
roc_png = OUT_DIR / "roc_real_1_2_4.png"
roc_pdf = OUT_DIR / "roc_real_1_2_4.pdf"
plt.savefig(roc_png, dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.savefig(roc_pdf, bbox_inches="tight")
plt.close()
print(f"Saved ROC figure:\n- {roc_png}\n- {roc_pdf}")


13. Real image horizontal error-map figure

In [ ]:
REAL_IMG_DIR = REAL_IMAGE_DIR
GT_MASK_DIR = REAL_MASK_DIR
OUT_DIR = REAL_OUTPUT_DIR

rows = matched_real_image_mask_rows(REAL_IMG_DIR, GT_MASK_DIR)
if not rows:
    raise FileNotFoundError(
        f"No matching image / mask pairs found. Check REAL_IMAGE_DIR={REAL_IMG_DIR} and REAL_MASK_DIR={GT_MASK_DIR}."
    )
letters = ["A", "B", "C"]

def resolve_with_any_ext(folder, stem):
    for ext in (".png",".jpg",".jpeg",".tif",".tiff",".bmp"):
        p = Path(folder)/f"{stem}{ext}"
        if p.exists(): return str(p)
    return None

def load_bin(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None: raise IOError(f"Cannot read {path}")
    if m.ndim==3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return (m>0).astype(np.uint8)

# Collect per-image error maps
err_maps = []
for img_stem, gt_stem in rows:
    gt_path   = resolve_with_any_ext(GT_MASK_DIR, gt_stem)
    pred_path = Path(OUT_DIR)/f"{img_stem}_mask.png"
    gt  = load_bin(gt_path)
    pred= load_bin(pred_path)

    if pred.shape != gt.shape:
        pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_NEAREST)

    pred_b = pred.astype(bool); gt_b = gt.astype(bool)
    TP = np.logical_and(pred_b, gt_b)
    FP = np.logical_and(pred_b, ~gt_b)
    FN = np.logical_and(~pred_b, gt_b)

    err = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)  # black canvas
    err[TP] = (0,255,0)    # green
    err[FP] = (255,0,0)    # red
    err[FN] = (0,0,255)    # blue

    err_maps.append(err)

# Plot horizontally (3 cols, 1 row), with panel letters
fig, axes = plt.subplots(1, len(err_maps), figsize=(12, 4))
for ax, im, letter in zip(axes, err_maps, letters):
    ax.imshow(im)
    ax.axis('off')
    ax.set_title(letter, fontsize=12, fontweight="bold", pad=4)

plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0.02)
out_path = Path(OUT_DIR)/"real_error_maps_dark_horizontal_lettered.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.close()

print("Saved horizontal error map figure →", out_path)
